# Four-Dimensional Rating System

This notebook extends the Elo framework into four complementary ratings:

1. **Seasonal Rating** — current-period competitive performance.
2. **Overall Rating** — long-term competitive strength.
3. **Prosperity Rating** — the quality of a participant's development pathway: progression, knowledge transfer, disciplined continuation, and controlled volatility.
4. **Thinkers Rating** — strategic behaviour: whether a participant repeatedly uses the same approach or evaluates outcomes and adapts when adaptation is justified.

The first two ratings remain performance-oriented Elo measures. Prosperity and Thinkers are experimental behavioural/process metrics and should be interpreted as indicators derived from recorded behaviour, not as objective measures of intelligence or personality.

> **Design principle:** changing a strategy is not automatically rewarded. Adaptation receives positive credit when the new approach produces an improvement; unnecessary or unsuccessful changes are penalized.

In [ ]:
import math
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INITIAL_RATING = 1500.0
K_FACTOR = 40.0
HOME_ADVANTAGE = 50.0
SEASON_CARRY = 0.25

print("Four-dimensional rating system initialized.")

## 1. Elo Foundation

The Elo component estimates the expected result from the current ratings. The **Overall Rating** is updated continuously, while the **Seasonal Rating** is maintained as a shorter-horizon measure and can be partially reset between seasons.

In [ ]:
def expected_score(rating_a, rating_b, home_advantage=0.0):
    adjusted_a = rating_a + home_advantage
    return 1 / (1 + 10 ** ((rating_b - adjusted_a) / 400))


def elo_update(rating_a, rating_b, actual_a, k=K_FACTOR, home_advantage=0.0):
    expected_a = expected_score(rating_a, rating_b, home_advantage)
    expected_b = 1 - expected_a

    new_a = rating_a + k * (actual_a - expected_a)
    new_b = rating_b + k * ((1 - actual_a) - expected_b)

    return new_a, new_b, expected_a


def season_regression(rating, baseline=INITIAL_RATING, carry=SEASON_CARRY):
    return baseline + carry * (rating - baseline)

## 2. Participant State

The additional fields record the behavioural information required for Prosperity and Thinkers ratings.

- `strategy`: the strategy used in the current match.
- `knowledge_transfer`: a normalized signal for whether useful knowledge/process was carried forward.
- `volatility`: a normalized measure of how much the participant changed approach.
- `matches`, `wins`, `strategy_history`: supporting history for analysis.

These fields are intentionally explicit so that the model can later be connected to real event logs rather than inferred from a final score alone.

In [ ]:
def create_participant(name):
    return {
        "name": name,
        "seasonal_rating": INITIAL_RATING,
        "overall_rating": INITIAL_RATING,
        "prosperity_rating": INITIAL_RATING,
        "thinkers_rating": INITIAL_RATING,
        "matches": 0,
        "wins": 0,
        "strategy_history": [],
    }


participants = {
    name: create_participant(name)
    for name in ["Alice", "Bob", "Charlie", "Diana"]
}

participants["Alice"]

## 3. Prosperity Rating

The Prosperity Rating represents a **development pathway** rather than match strength alone.

A positive update is driven by:

- improvement in competitive rating,
- transferring useful knowledge/process from one period to the next,
- controlled rather than chaotic volatility.

A participant can therefore prosper by following a disciplined path, preserving useful lessons, and changing approach when there is a reason to do so.

This is a modelling choice for the project, not a standard Elo statistic.

In [ ]:
def prosperity_update(current_prosperity, rating_change,
                      knowledge_transfer, volatility):
    # Scale competitive progress into a comparable process signal.
    progress_signal = np.tanh(rating_change / 100.0)

    # Knowledge transfer is rewarded; excessive volatility is penalized.
    delta = (
        35.0 * progress_signal
        + 20.0 * knowledge_transfer
        - 10.0 * volatility
    )

    return max(0.0, current_prosperity + delta)


def thinkers_update(current_thinkers, strategy_changed,
                    performance_change):
    # Adaptation is valuable when it improves performance.
    if strategy_changed and performance_change > 0:
        signal = 1.0
    elif not strategy_changed and performance_change >= 0:
        # Consistency is useful when the current approach is still working.
        signal = 0.5
    elif strategy_changed and performance_change <= 0:
        # Unsuccessful switching is treated as experimentation with a cost.
        signal = -0.5
    else:
        # Staying with a deteriorating approach is the weakest behaviour.
        signal = -0.25

    return max(0.0, current_thinkers + 25.0 * signal)

## 4. Example Match/Event Data

The example includes strategy labels and behavioural signals so the four-dimensional model has information beyond the match result.

`knowledge_transfer` is deliberately explicit. In a production system it could come from observed events such as reusing a successful pattern, documenting a lesson, teaching another participant, or carrying a proven process into the next period.

`volatility` should likewise come from measurable strategy/process changes rather than subjective judgement.

In [ ]:
matches = pd.DataFrame([
    ["2026-01-05", "Alice",   "Bob",     1,   "A", "A", 1.0, 0.0],
    ["2026-01-06", "Charlie", "Diana",   0,   "B", "B", 1.0, 0.0],
    ["2026-01-12", "Alice",   "Charlie", 0.5, "A", "B", 0.8, 0.6],
    ["2026-01-13", "Bob",     "Diana",   0,   "C", "B", 0.7, 0.5],
    ["2026-01-20", "Alice",   "Diana",   1,   "C", "B", 1.0, 0.8],
    ["2026-01-21", "Bob",     "Charlie", 1,   "C", "B", 0.9, 0.7],
    ["2026-02-03", "Alice",   "Bob",     0,   "C", "C", 1.0, 0.0],
    ["2026-02-04", "Charlie", "Diana",   1,   "B", "C", 0.9, 0.7],
], columns=[
    "date", "home", "away", "home_result",
    "home_strategy", "away_strategy",
    "knowledge_transfer", "volatility"
])

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)
matches

## 5. Process the Four Ratings

For each event:

- **Overall** receives the normal Elo update.
- **Seasonal** receives the same result update but represents the current period.
- **Prosperity** evaluates progression, knowledge transfer, and volatility.
- **Thinkers** evaluates whether strategy repetition or adaptation was appropriate.

The Thinkers score intentionally rewards both **successful consistency** and **successful adaptation**. This avoids the assumption that changing strategy is always smarter.

In [ ]:
def process_match(participants, row):
    home = participants[row["home"]]
    away = participants[row["away"]]

    home_old_overall = home["overall_rating"]
    away_old_overall = away["overall_rating"]
    home_old_seasonal = home["seasonal_rating"]
    away_old_seasonal = away["seasonal_rating"]

    # Overall Elo
    home["overall_rating"], away["overall_rating"], expected_home = elo_update(
        home_old_overall,
        away_old_overall,
        row["home_result"],
        k=K_FACTOR,
        home_advantage=HOME_ADVANTAGE,
    )

    # Seasonal Elo uses the same competitive signal but remains a separate state.
    home["seasonal_rating"], away["seasonal_rating"], _ = elo_update(
        home_old_seasonal,
        away_old_seasonal,
        row["home_result"],
        k=K_FACTOR,
        home_advantage=HOME_ADVANTAGE,
    )

    # Result-based rating change used by the process metrics.
    home_change = home["overall_rating"] - home_old_overall
    away_change = away["overall_rating"] - away_old_overall

    home_changed = bool(home["strategy_history"]) and (
        home["strategy_history"][-1] != row["home_strategy"]
    )
    away_changed = bool(away["strategy_history"]) and (
        away["strategy_history"][-1] != row["away_strategy"]
    )

    home["prosperity_rating"] = prosperity_update(
        home["prosperity_rating"],
        home_change,
        row["knowledge_transfer"],
        row["volatility"],
    )
    away["prosperity_rating"] = prosperity_update(
        away["prosperity_rating"],
        away_change,
        row["knowledge_transfer"],
        row["volatility"],
    )

    # A simple local performance signal: positive Elo movement means improvement.
    home["thinkers_rating"] = thinkers_update(
        home["thinkers_rating"], home_changed, home_change
    )
    away["thinkers_rating"] = thinkers_update(
        away["thinkers_rating"], away_changed, away_change
    )

    home["matches"] += 1
    away["matches"] += 1

    if row["home_result"] == 1:
        home["wins"] += 1
    elif row["home_result"] == 0:
        away["wins"] += 1

    home["strategy_history"].append(row["home_strategy"])
    away["strategy_history"].append(row["away_strategy"])

    return {
        "date": row["date"],
        "home": home["name"],
        "away": away["name"],
        "expected_home": expected_home,
        "home_overall": home["overall_rating"],
        "away_overall": away["overall_rating"],
    }


history = [process_match(participants, row) for _, row in matches.iterrows()]
history_df = pd.DataFrame(history)

history_df

## 6. Four-Dimensional Leaderboard

The four ratings answer different questions, so a single combined score is intentionally avoided.

| Rating | Question |
|---|---|
| Seasonal | Who is performing best right now? |
| Overall | Who has the strongest long-term record? |
| Prosperity | Who is building a disciplined, progressive pathway? |
| Thinkers | Who appropriately persists, evaluates, and adapts? |

In [ ]:
leaderboard = pd.DataFrame(participants.values())[
    [
        "name",
        "seasonal_rating",
        "overall_rating",
        "prosperity_rating",
        "thinkers_rating",
        "matches",
        "wins",
    ]
].sort_values("overall_rating", ascending=False).reset_index(drop=True)

leaderboard

## 7. Compare the Four Dimensions

The visualization makes it possible to distinguish competitive strength from behavioural/process characteristics.

In [ ]:
plot_df = leaderboard.set_index("name")[
    ["seasonal_rating", "overall_rating", "prosperity_rating", "thinkers_rating"]
]

plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Four-Dimensional Rating Profile")
plt.xlabel("Participant")
plt.ylabel("Rating")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Seasonal Transition

At the end of a season, the Seasonal Rating can regress toward the baseline while the Overall Rating remains intact.

For example, with a 25% carry factor:

`new seasonal = 1500 + 0.25 × (old seasonal − 1500)`

This preserves some information from the previous season while making the Seasonal Rating responsive to a new competitive period.

In [ ]:
before_after = []

for person in participants.values():
    old = person["seasonal_rating"]
    person["seasonal_rating"] = season_regression(old)
    before_after.append({
        "name": person["name"],
        "seasonal_before": old,
        "seasonal_after": person["seasonal_rating"],
        "overall": person["overall_rating"],
        "prosperity": person["prosperity_rating"],
        "thinkers": person["thinkers_rating"],
    })

pd.DataFrame(before_after).sort_values("overall", ascending=False)

## 9. Interpreting the System

### Seasonal Rating
Measures short-term form. It should be the most responsive of the performance ratings.

### Overall Rating
Measures accumulated competitive strength. It should be the primary rating when long-term ranking is required.

### Prosperity Rating
Measures whether a participant's pathway shows progression, knowledge retention/transfer, disciplined continuity, and controlled volatility. A high Prosperity Rating does **not** necessarily mean the participant has the highest Elo.

### Thinkers Rating
Measures the relationship between strategy persistence and adaptation. A high score should indicate that the participant tends to keep a strategy when it works and change it when evidence suggests that change is beneficial.

### Important modelling limitation
Prosperity and Thinkers depend on behavioural metadata. Without reliable observations of strategy, learning, knowledge transfer, and outcomes after changes, these scores can become arbitrary. For a serious implementation, those signals should come from event logs and be validated against historical data.

## 10. Suggested Production Extensions

1. Track strategy IDs and strategy versions instead of free-text labels.
2. Record the reason for each strategy change.
3. Measure performance for a fixed window before and after a change.
4. Track whether knowledge is reused successfully in later events.
5. Separate healthy experimentation from random volatility.
6. Add recovery metrics after a poor result.
7. Evaluate ratings against held-out historical data.
8. Calibrate Prosperity and Thinkers weights using empirical outcomes rather than fixed coefficients.
9. Keep behavioural ratings separate from any claim about intelligence, personality, or human worth.
10. Store a complete rating history so each participant can be plotted across seasons.

The four-dimensional model can therefore evolve from a conceptual notebook into a measurable ranking framework once the required behavioural events are available.